# Ridge & Lasso Regression (simple, as per slides)

**Ridge:** minimize `RSS + λ Σ βj²`  →  `β̂ = (XᵀX + λI)⁻¹ Xᵀy`

**Lasso:** minimize `RSS + λ Σ |βj|`  (no closed form)

In [ ]:
import numpy as np
np.random.seed(0)

## Data

In [ ]:
N, p = 100, 10
X = np.random.randn(N, p)
true_beta = np.array([5, -3, 2, 0, 0, 0, 0, 0, 0, 0], dtype=float)
y = 4 + X @ true_beta + 0.5 * np.random.randn(N)     # intercept = 4

# standardize X (so β0 = mean of y)
mu, sd = X.mean(0), X.std(0)
X = (X - mu) / sd

## Ridge Algorithm
1. Choose λ > 0
2. Create X, β, y
3. β̂ = (XᵀX + λI)⁻¹ (Xᵀy)
4. β̂₀ = (1/N) Σ yᵢ
5. x̃₀ = [1, x₀₁, ..., x₀p]
6. β̃ = [β̂₀, β̂]
7. Return β̃ᵀ · x̃₀

In [ ]:
def ridge_fit(X, y, lam):
    p = X.shape[1]
    beta = np.linalg.inv(X.T @ X + lam * np.eye(p)) @ (X.T @ y)   # step 3
    beta0 = y.mean()                                               # step 4
    return np.concatenate([[beta0], beta])                         # step 6  (β̃)

def predict(beta_tilde, x0):
    x0_tilde = np.concatenate([[1], (x0 - mu) / sd])               # step 5
    return beta_tilde @ x0_tilde                                   # step 7

lam = 10                                                           # step 1
beta_ridge = ridge_fit(X, y, lam)
print("Ridge β̃:", np.round(beta_ridge, 3))

## Lasso
Cost:  `J(β) = (1/2n) Σ(yᵢ − ŷᵢ)² + λ Σ|βj|`

Gradient:  `∂J/∂β = −(1/n) Xᵀ(y − Xβ) + λ·sign(β)`

Update:  `β = β − lr · ∂J/∂β`

(`|β|` never lands exactly on 0 with gradient steps, so tiny values `< 0.05` are set to 0 at the end.)

In [ ]:
def lasso_fit(X, y, lam, lr=0.05, iters=5000):
    n, p = X.shape
    beta0 = y.mean()
    yc = y - beta0
    beta = np.zeros(p)
    for _ in range(iters):
        grad = -(1/n) * X.T @ (yc - X @ beta) + lam * np.sign(beta)
        beta = beta - lr * grad
    beta[np.abs(beta) < 0.05] = 0
    return np.concatenate([[beta0], beta])

beta_lasso = lasso_fit(X, y, lam=0.3)
print("Lasso β̃:", np.round(beta_lasso, 3))

## Compare + predict a new point

In [ ]:
print("True    :", true_beta)
print("Ridge   :", np.round(beta_ridge[1:], 2))
print("Lasso   :", np.round(beta_lasso[1:], 2))
print("Lasso zeros:", np.sum(beta_lasso[1:] == 0), "of", p)

x0 = np.random.randn(p)
print("\nPrediction  Ridge:", round(predict(beta_ridge, x0), 3),
      "| Lasso:", round(predict(beta_lasso, x0), 3),
      "| True:", round(4 + x0 @ true_beta, 3))